# AIxCC SoK — CRS Architecture Walkthrough

**Paper:** *SoK: DARPA's AI Cyber Challenge (AIxCC): Competition Design, Architectures, and Lessons Learned*  
**arxiv:** https://arxiv.org/abs/2602.07666  
**Authors:** Zhang, Park, Fleischer, Fu, Kim (×3), Xu, Chin, Sheng, Zhao, Pelican, Musliner, Huang, Silliman, Mcdaniel, Casavant, Goldthwaite, Vidovich, Lehman, Kim

This notebook walks through the CRS architecture extracted from §3–§6.4 of the paper.
Each section quotes the paper directly, shows the corresponding scaffold code, and
runs a sanity check to verify the data structures work as expected.

**No GPU required. No external data needed. Everything runs on CPU instantly.**

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'aixcc_sok'))

from crs.pipeline import (
    Challenge, ScanMode, SubmissionType,
    SCORE_RANGES, time_decay_factor, accuracy_multiplier,
)
from crs.taxonomy import FINALIST_TEAMS, technique_adoption_rate, CRSProfile
from crs.pov_generation import PoV, PoVDeduplicator, BugCandidate
from crs.patch_generation import Patch, MinimalPatchSetCalculator
from crs.sarif_validation import SARIFVerdict, SARIFAssessment, PoVCentricStrategy
from crs.bundling import Bundle

print('Imports OK')

---
## §3 — Competition Design: The Four Submission Types

> "The four CRS capabilities each address a real development moment:  
> Full Scan, Delta Scan, SARIF Review, Report Synthesis."  
> — §3

> "Each submission's weight reflects how much developer time and effort it saves (or wastes, when wrong):  
> PoV: [1, 2] pts  
> Patch: [3, 6] pts  
> SARIF: [0.5, 1] pt  
> Bundle: [−7, 7] pts"  
> — §3

In [ ]:
# Verify score ranges match §3
for stype, srange in SCORE_RANGES.items():
    print(f'{stype.value:8s}: [{srange.min_score}, {srange.max_score}]')

assert SCORE_RANGES[SubmissionType.POV].min_score == 1.0
assert SCORE_RANGES[SubmissionType.POV].max_score == 2.0
assert SCORE_RANGES[SubmissionType.PATCH].min_score == 3.0
assert SCORE_RANGES[SubmissionType.PATCH].max_score == 6.0
assert SCORE_RANGES[SubmissionType.SARIF].min_score == 0.5
assert SCORE_RANGES[SubmissionType.BUNDLE].min_score == -7.0
print('\n✓ Score ranges match §3')

---
## §3 — Scoring: Time Decay and Accuracy Multiplier

> "Time-decay grants full points for immediate submissions and half for last-minute ones."  
> — §3

> "The accuracy multiplier penalizes a CRS's per-challenge total by its accuracy rate:  
> high accuracy barely affected (90% → negligible penalty),  
> while low accuracy is steeply penalized (50% → 6% reduction; 40% → 13% reduction)."  
> — §3

> **[UNSPECIFIED]** Full formula is in competition rulebook [9], not reproduced in paper.  
> We use linear interpolation from the three stated data points.

In [ ]:
# §3 — Time decay: 1.0 at start, 0.5 at end
total = 3600.0  # 1 hour challenge window
decay_immediate = time_decay_factor(0, total)
decay_midpoint = time_decay_factor(total/2, total)
decay_lastminute = time_decay_factor(total, total)

print('Time-decay factors (§3):')
print(f'  Immediate (t=0):   {decay_immediate:.3f}  — expect 1.000')
print(f'  Midpoint  (t=T/2): {decay_midpoint:.3f}  — expect 0.750 (linear)')
print(f'  Last-minute (t=T): {decay_lastminute:.3f}  — expect 0.500')

assert decay_immediate == 1.0,  f'Expected 1.0, got {decay_immediate}'
assert decay_lastminute == 0.5, f'Expected 0.5, got {decay_lastminute}'
print('✓ Time decay boundaries match §3')

print('\nAccuracy multiplier (§3 data points):')
for acc in [1.0, 0.9, 0.5, 0.4]:
    mult = accuracy_multiplier(acc)
    reduction_pct = (1.0 - mult) * 100
    print(f'  accuracy={acc:.1%} → multiplier={mult:.4f} ({reduction_pct:.1f}% reduction)')

# §3 data points: 90%→negligible, 50%→6%, 40%→13%
assert abs(accuracy_multiplier(0.5) - 0.94) < 0.01, '50% accuracy should give ~6% reduction'
assert abs(accuracy_multiplier(0.4) - 0.87) < 0.01, '40% accuracy should give ~13% reduction'
print('✓ Accuracy multiplier matches §3 data points')

---
## §5, Tables 2–6 — Taxonomy: Seven Finalist CRS Teams

> "While all teams built systems targeting the same CRS core capabilities (§3),  
> their architectural approaches varied widely, shaped by team expertise,  
> resource constraints, and strategic priorities."  
> — §5

In [ ]:
# Verify taxonomy has all 7 finalist teams (§5, Table 2)
assert len(FINALIST_TEAMS) == 7, f'Expected 7 teams, got {len(FINALIST_TEAMS)}'

print('Finalist CRS teams (§5, Table 2) — ordered by final rank:')
for team_id, profile in sorted(FINALIST_TEAMS.items(), key=lambda x: x[1].metadata.final_rank):
    m = profile.metadata
    print(f'  [{m.final_rank}] {m.id:3s} | {m.crs_name:15s} | {m.background:10s} | {m.llm_lib}')

print('\n✓ All 7 finalist teams present')

In [ ]:
# Technique adoption rates from §6.1, Table 3
print('PoV Generation technique adoption rates (§6.1, Table 3):')
pov_techniques = [
    ('pov.fuzzing.parallel_fuzzing',        '§6.1 — "all teams run multiple fuzzer instances in parallel"'),
    ('pov.fuzzing.pre_competition_corpus',  '§6.1 — "five teams reused pre-collected corpora"'),
    ('pov.fuzzing.seed_gen_agent',          '§6.1 — "six teams use LLMs to generate seeds"'),
    ('pov.llm_pipeline.pov_gen_agent',      '§6.1 — "five teams construct PoV-generation agents"'),
    ('pov.llm_pipeline.cwe_guidance',       '§6.1 — "four teams inject CWE-specific guidance"'),
    ('pov.llm_pipeline.reach_then_exploit', '§6.1 — "three teams: reach agent + exploit agent"'),
    ('pov.cooperation.llm_to_fuzz',         '§6.1 — "five teams: LLM→Fuzz cooperation"'),
    ('pov.cooperation.fuzz_to_llm',         '§6.1 — "three teams: Fuzz→LLM cooperation"'),
]

for path, spec in pov_techniques:
    rate = technique_adoption_rate(path)
    count = round(rate * 7)
    bar = '█' * count + '░' * (7 - count)
    print(f'  {bar} {count}/7 {path.split(".")[-1]}')

# §6.1 — "all teams run parallel fuzzing"
assert technique_adoption_rate('pov.fuzzing.parallel_fuzzing') == 1.0
# §6.1 — "five teams explored both pipelines"
assert technique_adoption_rate('pov.llm_pipeline.pov_gen_agent') == 5/7
print('\n✓ Adoption rates match §6.1 prose')

---
## §6.1 — PoV Generation: Deduplication

> "All teams implement deduplication using crash stack traces, input hashing,  
> sanitizer signatures, etc."  
> — §6.1, Table 3 (PoV Submission rows)

> "TI and FB further use LLMs to group semantically equivalent PoVs."  
> — §6.1

In [ ]:
# Sanity check: PoV deduplication
dedup = PoVDeduplicator(use_llm_dedup=False)

pov_a = PoV(
    pov_id='pov-1',
    input_bytes=b'AAAA\x00\x00',
    crash_stack='#0 0x4004a1 in vuln_func /src/main.c:42\n#1 0x4005b2 in main',
    sanitizer_signature='ASAN:heap-buffer-overflow',
    source='fuzzing',
)

# Duplicate: same crash stack, different input bytes
pov_b = PoV(
    pov_id='pov-2',
    input_bytes=b'BBBB\x00\x00',  # different bytes
    crash_stack='#0 0x4004a1 in vuln_func /src/main.c:42\n#1 0x4005b2 in main',  # same stack
    source='llm',
)

# Different crash — genuinely new PoV
pov_c = PoV(
    pov_id='pov-3',
    input_bytes=b'CCCC',
    crash_stack='#0 0x400b12 in parse_input /src/parser.c:88',
    source='llm',
)

dedup.register(pov_a)
assert dedup.is_duplicate(pov_b) is True,  'pov_b: same crash stack → should be duplicate'
assert dedup.is_duplicate(pov_c) is False, 'pov_c: different crash → should not be duplicate'

dedup.register(pov_c)
# Now pov_c's bytes are known
pov_c_copy = PoV(pov_id='pov-4', input_bytes=b'CCCC', source='fuzzing')
assert dedup.is_duplicate(pov_c_copy) is True, 'pov_c_copy: same input bytes → duplicate'

print('✓ PoVDeduplicator: crash_stack, sanitizer_signature, and input_hash dedup all work')

---
## §6.2 — Patch Generation: Minimal Patch Set

> "Four CRSs (AT, TB, SP, 42) leverage the fact that a single patch can fix  
> multiple PoVs sharing the same root cause, and compute a minimal patch set  
> that covers all known PoVs to avoid duplicate submissions."  
> — §6.2

> **[UNSPECIFIED]** Whether greedy or exact cover algorithm used.  
> We use greedy set cover (O(n log n), standard NP-hard approximation).

In [ ]:
# Minimal patch set sanity check
config = {'patch_generation': {'dedup_and_submit': {'patch_set_mode': 'full_recompute'}}}
calc = MinimalPatchSetCalculator(config)

# 3 PoVs, 3 candidate patches:
#   patch_x covers pov-1, pov-2 (root cause: buffer overflow in parse)
#   patch_y covers pov-2, pov-3 (partial overlap)
#   patch_z covers pov-3       (only pov-3)
patch_x = Patch(patch_id='patch-x', diff='-bad\n+good\n', source_pov_ids=['pov-1', 'pov-2'])
patch_y = Patch(patch_id='patch-y', diff='-old\n+new\n',  source_pov_ids=['pov-2', 'pov-3'])
patch_z = Patch(patch_id='patch-z', diff='-err\n+fix\n',  source_pov_ids=['pov-3'])

all_pov_ids = ['pov-1', 'pov-2', 'pov-3']
minimal = calc.compute([patch_x, patch_y, patch_z], all_pov_ids)

# Greedy should pick: patch_x (covers pov-1,2) then patch_z (covers pov-3) = 2 patches
# OR: patch_y (covers pov-2,3) then patch_x (still needs pov-1) — but patch_x is greedier first
covered = set()
for p in minimal:
    covered.update(p.source_pov_ids)

assert covered >= set(all_pov_ids), f'Minimal set does not cover all PoVs: {covered}'
assert len(minimal) <= len(all_pov_ids), 'Should not need more patches than PoVs'

print(f'✓ Minimal patch set: {len(minimal)} patches cover all {len(all_pov_ids)} PoVs')
for p in minimal:
    print(f'   patch_id={p.patch_id}  covers={p.source_pov_ids}')

---
## §6.3 — SARIF Validation: PoV-Centric Strategy

> "AT, TB, FB primarily rely on PoV matching, submitting Correct only when a match  
> is found and withholding unmatched reports."  
> — §6.3

> "FB additionally uses a fallback LLM judgement, but only submits Correct from it."  
> — §6.3

Default strategy: PoV-centric (conservative; minimizes false positive penalty risk).

In [ ]:
import asyncio

# Sanity check: PoV-centric SARIF validation
strategy = PoVCentricStrategy(llm_client=None, use_llm_fallback_correct_only=False)

# SARIF report pointing to a file that appears in our PoV's crash stack
sarif_matching = {
    'id': 'report-001',
    'file_path': '/src/main.c',
    'line_start': 42,
    'cwe': 'CWE-119',
}

sarif_nonmatching = {
    'id': 'report-002',
    'file_path': '/src/unrelated.c',
    'line_start': 10,
    'cwe': 'CWE-476',
}

pov_with_crash = PoV(
    pov_id='pov-1',
    input_bytes=b'test',
    crash_stack='#0 0x4004a1 in vuln_func /src/main.c:42',
)

context = {'povs': [pov_with_crash]}

result_match    = asyncio.get_event_loop().run_until_complete(
    strategy.assess(sarif_matching, context)
)
result_nomatch  = asyncio.get_event_loop().run_until_complete(
    strategy.assess(sarif_nonmatching, context)
)

assert result_match.verdict   == SARIFVerdict.CORRECT,  f'Matching SARIF should be Correct, got {result_match.verdict}'
assert result_nomatch.verdict == SARIFVerdict.WITHHOLD, f'Non-matching SARIF should be withheld, got {result_nomatch.verdict}'

print(f'✓ PoV-centric: matching report → {result_match.verdict.value}')
print(f'✓ PoV-centric: non-matching report → {result_nomatch.verdict.value} (withhold — conservative)')

---
## §6.4 — Bundling: Data Structure Verification

> "A bundle can contain any two of three pairings: PoV-Patch, PoV-SARIF, and Patch-SARIF,  
> to form a complete scoring bundle, while **any incorrect pairing will penalize the entire bundle**."  
> — §6.4

Bundle score range: [−7, 7] pts (§3 — largest risk/reward of any submission type).

In [ ]:
# Bundle data structure sanity check
b = Bundle(
    bundle_id='bundle-001',
    pov_id='pov-1',
    patch_id='patch-x',
    sarif_report_id='report-001',
    has_pov_patch=True,
    has_pov_sarif=True,
)

assert b.has_pov_patch, 'PoV-Patch pairing should be set'
assert b.has_pov_sarif, 'PoV-SARIF pairing should be set'
assert not b.has_patch_sarif, 'Patch-SARIF should be off (no No-PoV capability in default config)'

# Bundle score range from §3
bundle_range = SCORE_RANGES[SubmissionType.BUNDLE]
assert bundle_range.min_score == -7.0
assert bundle_range.max_score ==  7.0

print(f'✓ Bundle {b.bundle_id}: pov={b.pov_id} patch={b.patch_id} sarif={b.sarif_report_id}')
print(f'✓ Bundle score range: [{bundle_range.min_score}, {bundle_range.max_score}] pts (§3)')
print('   Note: any incorrect pairing penalizes the ENTIRE bundle (§6.4)')

---
## §5 — Taxonomy: Design Philosophy Comparison

> Table 2 — Seven finalist teams with widely varying architectural approaches:
> Ensemble-First (AT), Expertise-Driven (TB), Agentic (TI), Simple+Diverse (FB),
> Comprehensive (SP), Pragmatic (42), DSPy-Workflow (LC)

The taxonomy lets you compare any technique dimension across all teams.

In [ ]:
# Patch architecture patterns across all teams (§6.2, Table 4)
print('Patch Agent Architecture by team (§6.2, Table 4):')
print(f'{"Team":5s} {"Multi-Arch":12s} {"Multi-Agent":12s} {"Single-Agent":12s} {"Rank":5s}')
print('-' * 50)

for team_id, profile in sorted(FINALIST_TEAMS.items(), key=lambda x: x[1].metadata.final_rank):
    arch = profile.patch.agent_arch
    rank = profile.metadata.final_rank
    ma   = '✓' if arch.multi_arch   else ' '
    mag  = '✓' if arch.multi_agent  else ' '
    sa   = '✓' if arch.single_agent else ' '
    print(f'{team_id:5s} {ma:12s} {mag:12s} {sa:12s} {rank}')

# §6.2 — verify team architecture assignments
assert FINALIST_TEAMS['AT'].patch.agent_arch.multi_arch   is True
assert FINALIST_TEAMS['TB'].patch.agent_arch.multi_agent  is True
assert FINALIST_TEAMS['FB'].patch.agent_arch.single_agent is True
print('\n✓ Architecture assignments match §6.2, Table 4')

In [ ]:
# Summary: technique adoption rates at a glance
print('Key technique adoption rates across 7 finalist teams:')
print(f'{"Technique":<40s} {"Teams":>6s} {"Rate":>6s}')
print('-' * 55)

techniques = [
    ('pov.submission.asap_submission',           '§6.1 ASAP PoV submission'),
    ('pov.fuzzing.parallel_fuzzing',             '§6.1 Parallel fuzzing'),
    ('pov.fuzzing.pre_competition_corpus',       '§6.1 Pre-competition corpus'),
    ('pov.llm_pipeline.pov_gen_agent',           '§6.1 LLM PoV gen agent'),
    ('pov.llm_pipeline.cwe_guidance',            '§6.1 CWE guidance injection'),
    ('pov.llm_pipeline.reach_then_exploit',      '§6.1 Reach→exploit decomposition'),
    ('patch.rca.standalone_rca',                 '§6.2 Standalone RCA component'),
    ('patch.rca.multi_pov_rca',                  '§6.2 Multi-PoV RCA'),
    ('patch.generation.llm_reflection',          '§6.2 LLM reflection'),
    ('patch.dedup_submit.minimal_patch_set',     '§6.2 Minimal patch set calc'),
    ('patch.validation.post_patch_fuzz',         '§6.2 Post-patch fuzzing'),
    ('sarif.pov_centric',                        '§6.3 PoV-centric SARIF strategy'),
    ('sarif.llm_judge_centric',                  '§6.3 LLM-judge SARIF strategy'),
]

for path, label in techniques:
    rate = technique_adoption_rate(path)
    count = round(rate * 7)
    bar = '█' * count + '░' * (7 - count)
    print(f'{label:<40s} {bar} {count}/7 ({rate:.0%})')

---
## §8 — Common Pitfalls and Implementation Notes

### 1. PoV dedup strategy matters
§6.1 — Deduplication should use **stack trace first**, then sanitizer signature, then input hash.
Using only input hash is insufficient (same crash, different inputs). Stack trace captures
the semantic equivalence that matters for scoring.

### 2. Patch-SARIF is high-risk
§6.4 — Only TI and FB implemented Patch-SARIF. "Any incorrect pairing penalizes the entire bundle."
Keep Patch-SARIF disabled unless No-PoV patch generation is working reliably.

### 3. LLM reflection is high-value
§6.2 — 5/7 teams used LLM reflection for patch generation. TB's dedicated reflection agent
"analyzes failures at each generation step and provides corrective guidance." Don't skip this.

### 4. Minimal patch set prevents duplicate penalties
§6.2 — Without minimal patch set calculation, submitting one patch per PoV incurs
"many duplicate submissions and patch-score penalties." Implement this before scaling.

### 5. Bundle timing
§6.4 — Bundles can be **freely updated until the deadline** (no time-decay penalty).
Set `rebundle_on_new_result=True` and keep updating as more PoVs and patches arrive.

### 6. No-PoV patches: delay and gate
§6.2 — If implementing No-PoV patches, all three teams (TI, FB, LC) that used this
strategy delayed submission significantly (45 min, 50% of time, 30 min before end)
and gated on prior accuracy. Do not submit No-PoV patches immediately.

### 7. SARIF: start conservative
§6.3 — The PoV-centric strategy (AT=1st, TB=2nd) withholds verdicts rather than
submitting Incorrect. Start here before considering LLM-judge-centric.
Incorrect SARIF verdicts reduce accuracy rate, triggering the multiplier penalty.

### 8. Vibecoding works but review before submission
§5 — "over 90% of FB's codebase is vibecoded." FB finished 4th (above SP=5th who built 53 components).
Rapid LLM-generated code can be effective, but validate each component carefully before
connecting it to the live competition pipeline.

In [ ]:
print('All walkthrough cells completed successfully.')
print('\nImplementation checklist for a production CRS:')
print('  [ ] Replace NotImplementedError stubs with real LLM prompts')
print('  [ ] Wire in OSS-Fuzz runner (OSSFuzzFuzzingPipeline.run)')
print('  [ ] Implement code indexer (symbol lookup for patch context)')
print('  [ ] Implement SAST output parsers (CodeQL/Semgrep JSON)')
print('  [ ] Implement GDB/JDB integration (dynamic info for patch gen)')
print('  [ ] Populate CWE knowledge base (pov_generation.py _get_cwe_guidance)')
print('  [ ] Implement competition API client (challenge webhook + submission)')
print('  [ ] Test on a local OSS-Fuzz challenge project before live competition')